# Gaussian Mixture Models and EM — Exercises

Companion to the note [Gaussian Mixture Models and EM](Gaussian%20Mixture%20Models%20and%20EM.md).

Practise the GMM density, responsibilities, the EM updates and why they never decrease the likelihood, singularities, choosing $K$ with BIC, covariance types, and the link to k-means: by hand first, then a full EM implementation in NumPy checked against `sklearn.mixture.GaussianMixture`.

**16 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.stats import norm, multivariate_normal
from scipy.special import logsumexp
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score
rng = np.random.default_rng(0)
# Shared 2-D data: three Gaussian clusters with different spreads
X_g, y_g = make_blobs(500, centers=[[0, 0], [5, 5], [0, 6]], cluster_std=[1.0, 1.5, 0.7], random_state=3)

## Part A · Concepts

### Exercise 1 · Why no closed form?
*🧠 Concept · ★☆☆*

Write the log-likelihood of $N$ i.i.d. points under the GMM $p(x)=\sum_k\pi_k\mathcal N(x\mid\mu_k,\Sigma_k)$.
Why does setting its gradient to zero not give a closed-form solution, unlike a single Gaussian? How does introducing the latent $z$ help?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compare $\log\prod_i\mathcal N(x_i\mid\mu,\Sigma)$ with $\log\prod_i\sum_k(\dots)$: where does the log end up?

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\ell(\theta)=\sum_i\log\sum_k\pi_k\mathcal N(x_i\mid\mu_k,\Sigma_k)$. The **log of a sum** does not split into separate terms, so the gradient w.r.t. $\mu_k$ contains the ratio $\gamma_{ik}$, which itself depends on all parameters: a coupled non-linear system.
If we knew $z_i$, the complete-data log-likelihood $\sum_i\big[\log\pi_{z_i}+\log\mathcal N(x_i\mid\mu_{z_i},\Sigma_{z_i})\big]$ separates per component and has closed-form MLEs (counts, means, covariances).
EM replaces the unknown indicators by their posterior expectations $\gamma_{ik}$ (E-step) and then solves those easy weighted problems (M-step).

</details>

### Exercise 2 · Soft vs hard assignments
*🧠 Concept · ★☆☆*

The note says "k-means = GMM with equal spherical covariances and hard assignments". Name two practical advantages of the
soft, full-covariance GMM over k-means, and one disadvantage.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about what the output is (probabilities, a density) and the cluster shapes each can represent.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Advantages: (1) **calibrated uncertainty**: a point between two clusters gets $\gamma\approx(0.5,0.5)$ instead of a forced label; (2) **elliptical, differently sized clusters** via $\Sigma_k$ and $\pi_k$ (Exercise 15); (3) a proper **density** $p(x)$ usable for anomaly detection ([[Anomaly Detection]]), sampling and model selection with BIC.
Disadvantages: more parameters ($O(Kd^2)$), so it needs more data, is slower, can hit singularities, and is still sensitive to initialization ([[Clustering]]).

</details>

### Exercise 3 · Singularities and restarts
*🧠 Concept · ★★☆*

Explain why the GMM likelihood is **unbounded** (the note's "singularities"). Why doesn't this happen for a single Gaussian fitted to $N\ge d+1$ distinct points?
List the practical remedies from the note, and say why the local-optimum problem cannot be fixed by running EM longer.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Put $\mu_k=x_1$ and let $\sigma_k\to0$. What happens to $\mathcal N(x_1\mid\mu_k,\sigma_k^2I)$ and to the other terms?

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $\mu_k=x_1$, $\mathcal N(x_1\mid x_1,\sigma^2I)=(2\pi\sigma^2)^{-d/2}\to\infty$ as $\sigma\to0$, while every other point is still explained by the other components, so its density stays bounded below. Hence $\ell\to\infty$ (Exercise 13).
A single Gaussian must explain *all* points with the same $\Sigma$; shrinking it makes the other points' densities go to 0 faster, so the likelihood has a finite maximum.
Remedies: add a small ridge $\Sigma_k+\epsilon I$ (sklearn `reg_covar`), a prior on $\Sigma$ (MAP / Bayesian GMM), drop or reinitialize collapsing components.
Local optima: EM is a monotone ascent method that converges to *a* stationary point determined by the start, so running longer cannot escape. Use several restarts (`n_init`) and k-means initialization, and keep the best likelihood.

</details>

## Part B · By hand

### Exercise 4 · Responsibilities (E-step)
*✍️ By hand · ★☆☆*

1-D GMM: $\pi=(0.5,0.5)$, $\mu=(0,2)$, $\sigma_1=\sigma_2=1$. Compute the responsibility $\gamma_{1}$ of component 1 for $x=1.5$.
Then answer: for which $x$ is $\gamma_1=0.5$?

In [ ]:
gamma1 = None
x_equal = None

_p = 0.5 * norm.pdf(1.5, [0, 2], 1)
check('gamma_1(1.5)', gamma1, _p[0] / _p.sum())
check('point with equal responsibility', x_equal, 1.0)

<details>
<summary><b>💡 Hint</b></summary>

With equal weights and variances, $\gamma_1=\frac{1}{1+\exp\left(\frac{(x-\mu_1)^2-(x-\mu_2)^2}{2}\right)}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$(x-0)^2/2=1.125$ and $(x-2)^2/2=0.125$, so $\gamma_1=\dfrac{e^{-1.125}}{e^{-1.125}+e^{-0.125}}=\dfrac1{1+e}\approx0.269$.
By symmetry $\gamma_1=0.5$ at the midpoint $x=1$. With equal variances the responsibility is a logistic function of $x$, just like the posterior of a generative classifier.

```python
gamma1 = 1 / (1 + np.e)
x_equal = 1.0
```

</details>

### Exercise 5 · One M-step by hand
*✍️ By hand · ★☆☆*

Data $x=(0,1,3,4)$ with responsibilities for component 1: $\gamma_{\cdot1}=(1,\ 1,\ 0.5,\ 0)$.
Compute $N_1$, $\pi_1$, $\mu_1$ and $\sigma_1^2$ using the note's M-step.

In [ ]:
N1 = None
pi1 = None
mu1 = None
var1 = None

_x = np.array([0, 1, 3, 4.]); _g = np.array([1, 1, .5, 0])
check('N1', N1, _g.sum()); check('pi1', pi1, _g.mean())
check('mu1', mu1, np.average(_x, weights=_g))
check('var1', var1, np.average((_x - np.average(_x, weights=_g)) ** 2, weights=_g))

<details>
<summary><b>💡 Hint</b></summary>

Compute $\mu_1$ first; the variance uses the **new** $\mu_1$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$N_1=2.5$, $\pi_1=2.5/4=0.625$, $\mu_1=(0+1+1.5+0)/2.5=1.0$,
$\sigma_1^2=\big(1\cdot1^2+1\cdot0^2+0.5\cdot2^2+0\big)/2.5=3/2.5=1.2$.

```python
N1 = 2.5
pi1 = 0.625
mu1 = 1.0
var1 = 1.2
```

</details>

### Exercise 6 · Deriving the M-step for the means
*✍️ By hand · ★★☆*

The M-step maximizes the expected complete-data log-likelihood
$Q(\theta)=\sum_i\sum_k\gamma_{ik}\big[\log\pi_k+\log\mathcal N(x_i\mid\mu_k,\Sigma_k)\big]$.
Derive $\mu_k$ and, with a Lagrange multiplier for $\sum_k\pi_k=1$, derive $\pi_k$.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint 1</b></summary>

$\nabla_\mu\log\mathcal N(x\mid\mu,\Sigma)=\Sigma^{-1}(x-\mu)$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

For $\pi$: maximize $\sum_k N_k\log\pi_k+\lambda(1-\sum_k\pi_k)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\nabla_{\mu_k}Q=\sum_i\gamma_{ik}\Sigma_k^{-1}(x_i-\mu_k)=0\Rightarrow\mu_k=\frac{\sum_i\gamma_{ik}x_i}{\sum_i\gamma_{ik}}=\frac1{N_k}\sum_i\gamma_{ik}x_i$.
For the weights: $\partial/\partial\pi_k\big[\sum_jN_j\log\pi_j+\lambda(1-\sum_j\pi_j)\big]=N_k/\pi_k-\lambda=0\Rightarrow\pi_k=N_k/\lambda$; summing over $k$ gives $\lambda=\sum_kN_k=N$, so $\pi_k=N_k/N$.
(The covariance update follows the same way from $\nabla_{\Sigma^{-1}}$.) Note the difference to direct MLE: here $\gamma_{ik}$ is held **fixed**.

</details>

### Exercise 7 · Mean and variance of a mixture
*✍️ By hand · ★★☆*

1-D mixture $\pi=(0.3,0.7)$, $\mu=(0,5)$, $\sigma^2=(1,2)$. Use the laws of total expectation and variance to compute $\mathbb E[x]$ and $\mathrm{Var}[x]$.

In [ ]:
mix_mean = None
mix_var = None

_t = np.linspace(-15, 25, 200001); _p = 0.3 * norm.pdf(_t, 0, 1) + 0.7 * norm.pdf(_t, 5, np.sqrt(2))
_m = np.trapezoid(_t * _p, _t) if hasattr(np, 'trapezoid') else np.trapz(_t * _p, _t)
_v = (np.trapezoid if hasattr(np, 'trapezoid') else np.trapz)((_t - _m) ** 2 * _p, _t)
check('mean (numerical integration)', mix_mean, _m, tol=1e-4)
check('variance (numerical integration)', mix_var, _v, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

$\mathrm{Var}[x]=\mathbb E[\mathrm{Var}[x\mid z]]+\mathrm{Var}[\mathbb E[x\mid z]]$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathbb E[x]=0.3\cdot0+0.7\cdot5=3.5$.
$\mathbb E[\mathrm{Var}[x\mid z]]=0.3\cdot1+0.7\cdot2=1.7$; $\mathrm{Var}[\mathbb E[x\mid z]]=0.3(0-3.5)^2+0.7(5-3.5)^2=3.675+1.575=5.25$.
$\mathrm{Var}[x]=1.7+5.25=6.95$. The between-component spread usually dominates: the mixture is much wider than either component.

```python
mix_mean = 3.5
mix_var = 6.95
```

</details>

### Exercise 8 · Counting parameters for BIC
*✍️ By hand · ★★☆*

A full-covariance GMM in $d=2$ dimensions with $K=3$ components is fitted to $N=200$ points and reaches log-likelihood $\ell=-500$.
Count the free parameters $p$ and compute $\mathrm{BIC}=-2\ell+p\ln N$.

In [ ]:
p_free = None
bic_val = None

_gm = GaussianMixture(3, covariance_type='full').fit(X_g[:200])
check('free parameters (sklearn _n_parameters)', p_free, _gm._n_parameters())
check('BIC', bic_val, 1000 + _gm._n_parameters() * np.log(200))

<details>
<summary><b>💡 Hint</b></summary>

Weights: $K-1$ (they sum to one). Means: $Kd$. A symmetric $d\times d$ covariance has $d(d+1)/2$ free entries.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p=(K-1)+Kd+K\frac{d(d+1)}2=2+6+9=17$. $\mathrm{BIC}=1000+17\ln200\approx1000+17\cdot5.298=1090.07$.
Lower BIC is better; the $p\ln N$ term penalizes extra components more strongly than AIC's $2p$.

```python
p_free = 17
bic_val = 1000 + 17 * np.log(200)
```

</details>

### Exercise 9 · Why EM never decreases the likelihood
*✍️ By hand · ★★★*

For any distribution $q(z)$ over the latent variables show
$$\log p(x\mid\theta)=\underbrace{\mathbb E_q[\log p(x,z\mid\theta)]-\mathbb E_q[\log q(z)]}_{\mathcal L(q,\theta)}+D_{KL}\big(q(z)\,\Vert\,p(z\mid x,\theta)\big).$$
Use it to prove $\log p(x\mid\theta^{new})\ge\log p(x\mid\theta^{old})$ for one EM iteration. Connect to the ELBO of [[Variational Inference]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint 1</b></summary>

Write $\log p(x\mid\theta)=\log p(x,z\mid\theta)-\log p(z\mid x,\theta)$, add and subtract $\log q(z)$, and take $\mathbb E_q$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

E-step: choose $q=p(z\mid x,\theta^{old})$, making the KL zero. M-step: increase $\mathcal L$ in $\theta$. KL $\ge0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\log p(x\mid\theta)=\mathbb E_q\big[\log\tfrac{p(x,z\mid\theta)}{q(z)}\big]+\mathbb E_q\big[\log\tfrac{q(z)}{p(z\mid x,\theta)}\big]=\mathcal L(q,\theta)+D_{KL}(q\Vert p(z\mid x,\theta))$ (the left side does not depend on $z$).
E-step: $q=p(z\mid x,\theta^{old})$ makes KL $=0$, so $\mathcal L(q,\theta^{old})=\log p(x\mid\theta^{old})$ (the bound touches).
M-step: $\theta^{new}=\arg\max_\theta\mathcal L(q,\theta)$, and $\mathcal L(q,\theta)$ differs from $Q(\theta)$ only by the entropy of $q$, constant in $\theta$.
Then $\log p(x\mid\theta^{new})=\mathcal L(q,\theta^{new})+D_{KL}\ge\mathcal L(q,\theta^{new})\ge\mathcal L(q,\theta^{old})=\log p(x\mid\theta^{old})$.
$\mathcal L$ is exactly the ELBO; EM is coordinate ascent on it, with an exact E-step. Variational inference uses the same bound when the exact posterior is intractable.

</details>

## Part C · Code from scratch

### Exercise 10 · Multivariate Gaussian log-density
*💻 Code · ★☆☆*

Implement `log_gauss(X, mu, Sigma)` returning $\log\mathcal N(x_i\mid\mu,\Sigma)$ for all rows of $X$, using a **Cholesky factor**
instead of `inv` and `det`. Compare with `scipy.stats.multivariate_normal.logpdf`.

In [ ]:
def log_gauss(X, mu, Sigma):
    # TODO: L = cholesky(Sigma); solve L y = (x - mu); log det = 2 sum log diag(L)
    return None

Sig_t = np.array([[2.0, 0.6], [0.6, 1.0]]); mu_t = np.array([1.0, -1.0])
lg = log_gauss(X_g[:10], mu_t, Sig_t)

check('log density', lg, multivariate_normal(mu_t, Sig_t).logpdf(X_g[:10]))

<details>
<summary><b>💡 Hint</b></summary>

$\log\mathcal N=-\tfrac d2\log2\pi-\tfrac12\log|\Sigma|-\tfrac12\lVert L^{-1}(x-\mu)\rVert^2$ with $\Sigma=LL^\top$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Cholesky is about twice as cheap as a general factorization, numerically stable, and fails loudly if $\Sigma$ is not positive definite (e.g. a collapsing component).

```python
from scipy.linalg import solve_triangular
def log_gauss(X, mu, Sigma):
    L = np.linalg.cholesky(Sigma)
    Y = solve_triangular(L, (X - mu).T, lower=True)
    d = X.shape[1]
    return -0.5 * d * np.log(2 * np.pi) - np.log(np.diag(L)).sum() - 0.5 * (Y ** 2).sum(0)

Sig_t = np.array([[2.0, 0.6], [0.6, 1.0]]); mu_t = np.array([1.0, -1.0])
lg = log_gauss(X_g[:10], mu_t, Sig_t)
```

</details>

### Exercise 11 · A numerically stable E-step
*💻 Code · ★★☆*

Implement `e_step(X, pis, mus, Sigmas)` returning `(gamma, loglik)`: compute the $N\times K$ matrix
$\log\pi_k+\log\mathcal N(x_i\mid\mu_k,\Sigma_k)$, normalize rows with **logsumexp**, and return the total log-likelihood.
Test it with the parameters of a fitted `GaussianMixture`. Why is logsumexp necessary?

In [ ]:
def e_step(X, pis, mus, Sigmas):
    # TODO
    return None, None

gm_ref = GaussianMixture(3, random_state=0).fit(X_g)
gam, ll = e_step(X_g, gm_ref.weights_, gm_ref.means_, gm_ref.covariances_)

check('responsibilities', gam, gm_ref.predict_proba(X_g))
check('log-likelihood', ll, gm_ref.score(X_g) * len(X_g))

<details>
<summary><b>💡 Hint</b></summary>

`logsumexp(A, axis=1, keepdims=True)` from `scipy.special`; `gamma = np.exp(A - lse)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For a point far from all components every $\mathcal N(x_i\mid\cdot)$ underflows to 0 in floating point (e.g. $e^{-800}$), giving $0/0$. Subtracting the row maximum inside logsumexp keeps the largest term at $e^0=1$, so nothing underflows.

```python
def e_step(X, pis, mus, Sigmas):
    A = np.column_stack([np.log(p) + log_gauss(X, m, S) for p, m, S in zip(pis, mus, Sigmas)])
    lse = logsumexp(A, axis=1, keepdims=True)
    return np.exp(A - lse), lse.sum()

gm_ref = GaussianMixture(3, random_state=0).fit(X_g)
gam, ll = e_step(X_g, gm_ref.weights_, gm_ref.means_, gm_ref.covariances_)
```

</details>

### Exercise 12 · Full EM for a GMM
*💻 Code · ★★☆*

Implement `m_step(X, gamma, reg=1e-6)` (the note's updates, adding `reg` to the diagonal of each $\Sigma_k$) and
`fit_gmm(X, K, n_iter)` that initializes from k-means (means = centroids, covariances = identity, equal weights) and alternates
E and M steps, recording the log-likelihood each iteration in a list. Run it on `X_g` with $K=3$ and store the history in `ll_hist`.

In [ ]:
def m_step(X, gamma, reg=1e-6):
    # TODO: return pis, mus, Sigmas
    return None, None, None

def fit_gmm(X, K, n_iter=100, seed=0):
    # TODO: return (pis, mus, Sigmas), ll_history
    return None, None

params_g, ll_hist = fit_gmm(X_g, 3)

_best = GaussianMixture(3, n_init=5, random_state=0).fit(X_g)
check('log-likelihood never decreases', None if ll_hist is None else bool(np.all(np.diff(ll_hist) > -1e-8)), True)
check('reaches sklearn optimum (mean log-lik)', None if ll_hist is None else ll_hist[-1] / len(X_g), _best.score(X_g), tol=1e-4)

<details>
<summary><b>💡 Hint 1</b></summary>

Weighted covariance: `D = X - mu; S = (gamma[:, k, None] * D).T @ D / Nk`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Use your `e_step` for the log-likelihood; it is computed with the parameters *before* the M-step.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The log-likelihood increases monotonically (Exercise 9) and plateaus after a few dozen iterations at the same optimum sklearn finds.
In practice one stops when the improvement falls below a tolerance (sklearn: `tol=1e-3` on the mean log-likelihood).

```python
def m_step(X, gamma, reg=1e-6):
    Nk = gamma.sum(0)
    pis = Nk / len(X)
    mus = gamma.T @ X / Nk[:, None]
    Sigmas = []
    for k in range(gamma.shape[1]):
        D = X - mus[k]
        Sigmas.append((gamma[:, k, None] * D).T @ D / Nk[k] + reg * np.eye(X.shape[1]))
    return pis, mus, np.array(Sigmas)

def fit_gmm(X, K, n_iter=100, seed=0):
    km = KMeans(K, n_init=10, random_state=seed).fit(X)
    pis, mus = np.full(K, 1 / K), km.cluster_centers_
    Sigmas = np.array([np.eye(X.shape[1])] * K)
    hist = []
    for _ in range(n_iter):
        gamma, ll = e_step(X, pis, mus, Sigmas)
        hist.append(ll)
        pis, mus, Sigmas = m_step(X, gamma)
    return (pis, mus, Sigmas), hist

params_g, ll_hist = fit_gmm(X_g, 3)
print(ll_hist[0], ll_hist[-1])
```

</details>

### Exercise 13 · Watching a component collapse
*💻 Code · ★★☆*

Take 1-D data `x1 = rng.normal(size=50)` and a 2-component mixture with $\pi=(0.5,0.5)$, component 1 = $\mathcal N(0,1)$ fixed and
component 2 = $\mathcal N(x_1[0],\sigma^2)$ centred **on a data point**. Compute the total log-likelihood for
$\sigma\in\{10^{-1},10^{-3},10^{-6},10^{-12},10^{-24}\}$ and store it in `ll_collapse`. Then repeat with the variance floored at `reg = 1e-3` (i.e. $\sigma^2+\text{reg}$) in `ll_reg`.

In [ ]:
x1 = rng.normal(size=50)
sigmas = [1e-1, 1e-3, 1e-6, 1e-12, 1e-24]
ll_collapse = None
ll_reg = None

check('likelihood grows without bound', None if ll_collapse is None else bool(np.all(np.diff(ll_collapse[1:]) > 0) and ll_collapse[-1] - ll_collapse[0] > 20), True)
check('regularized likelihood stays bounded', None if ll_reg is None else bool(max(ll_reg) < ll_collapse[-1] - 5), True)

<details>
<summary><b>💡 Hint</b></summary>

Use `norm.logpdf` and `np.logaddexp` for the log of a two-term sum.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Once $\sigma$ is small enough that component 2 no longer helps the other points, each tenfold reduction of $\sigma$ adds $\ln10\approx2.3$ to the log-likelihood through the one point the component sits on, so $\ell\to\infty$: the "MLE" is a degenerate spike.
With a variance floor the term $-\tfrac12\log(\sigma^2+\text{reg})$ is bounded and the problem goes away. This is why sklearn adds `reg_covar=1e-6` to every covariance.

```python
x1 = rng.normal(size=50)
sigmas = [1e-1, 1e-3, 1e-6, 1e-12, 1e-24]
def mix_ll(var2, reg=0.0):
    a = np.log(0.5) + norm.logpdf(x1, 0, 1)
    b = np.log(0.5) + norm.logpdf(x1, x1[0], np.sqrt(var2 + reg))
    return np.logaddexp(a, b).sum()
ll_collapse = [mix_ll(s ** 2) for s in sigmas]
ll_reg = [mix_ll(s ** 2, reg=1e-3) for s in sigmas]
print(np.round(ll_collapse, 1)); print(np.round(ll_reg, 1))
```

</details>

### Exercise 14 · Choosing K with BIC
*💻 Code · ★★☆*

Fit `GaussianMixture(K, n_init=3, random_state=0)` for $K=1,\dots,6$ on `X_g`, compute BIC and AIC, plot them,
and store the BIC-optimal $K$ in `K_bic`. Why can't you choose $K$ by the training log-likelihood?

In [ ]:
Ks = range(1, 7)
K_bic = None

check('BIC picks the true K', K_bic, len(np.unique(y_g)))

<details>
<summary><b>💡 Hint</b></summary>

`gm.bic(X)` and `gm.aic(X)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

BIC is minimal at $K=3$, the true number of clusters. The training log-likelihood can only go up with $K$ (more components can always fit at least as well), so it would always choose the largest model.
BIC approximates $-2\log p(D\mid M)$, the model evidence ([[Bayesian Inference]]); AIC penalizes less and tends to choose larger $K$. Cross-validated held-out log-likelihood is another option ([[Cross-Validation and Model Selection]]).

```python
import matplotlib.pyplot as plt
Ks = range(1, 7)
models = [GaussianMixture(K, n_init=3, random_state=0).fit(X_g) for K in Ks]
bics = [m.bic(X_g) for m in models]; aics = [m.aic(X_g) for m in models]
plt.plot(Ks, bics, 'o-', label='BIC'); plt.plot(Ks, aics, 's-', label='AIC'); plt.legend(); plt.xlabel('K'); plt.show()
K_bic = list(Ks)[int(np.argmin(bics))]
```

</details>

### Exercise 15 · k-means vs GMM on stretched clusters
*💻 Code · ★★☆*

Stretch blobs with a linear map (below) so the clusters become elongated ellipses. Cluster them with `KMeans(3)` and a full-covariance
`GaussianMixture(3, n_init=5, random_state=0)`; store the ARIs in `ari_km` and `ari_gmm` and plot both labelings.

In [ ]:
X_an, y_an = make_blobs(600, random_state=170)
X_an = X_an @ np.array([[0.6, -0.6], [-0.4, 0.8]])
ari_km = None
ari_gmm = None

check('GMM handles elongated clusters', None if ari_gmm is None else (ari_gmm > 0.95 and ari_km < 0.8), True)

<details>
<summary><b>💡 Hint</b></summary>

k-means' decision boundaries are perpendicular bisectors between centroids (a Voronoi diagram).

</details>

<details>
<summary><b>✅ Solution</b></summary>

k-means scores ARI $\approx0.58$: its Voronoi cells cut across the long thin clusters. The GMM learns each $\Sigma_k$ and recovers the clusters almost perfectly (ARI $\approx1$).
Standardizing the features would not fix this, because the elongation is not axis-aligned.

```python
import matplotlib.pyplot as plt
X_an, y_an = make_blobs(600, random_state=170)
X_an = X_an @ np.array([[0.6, -0.6], [-0.4, 0.8]])
lab_km = KMeans(3, n_init=10, random_state=0).fit_predict(X_an)
lab_gm = GaussianMixture(3, n_init=5, random_state=0).fit(X_an).predict(X_an)
ari_km, ari_gmm = adjusted_rand_score(y_an, lab_km), adjusted_rand_score(y_an, lab_gm)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.5))
ax[0].scatter(*X_an.T, c=lab_km, s=6); ax[0].set_title(f'k-means ARI={ari_km:.2f}')
ax[1].scatter(*X_an.T, c=lab_gm, s=6); ax[1].set_title(f'GMM ARI={ari_gmm:.2f}')
plt.show()
```

</details>

### Exercise 16 · Covariance types and their cost
*💻 Code · ★★★*

sklearn offers `covariance_type` in `{'spherical','diag','tied','full'}`. Write `n_params(K, d, cov_type)` returning the number of free
parameters for each, and verify against sklearn for $K=3,d=2$ and for $K=5,d=10$. Which type would you choose for $d=100$ with $N=500$?

In [ ]:
def n_params(K, d, cov_type):
    # TODO
    return None

ours = None if n_params(3, 2, 'full') is None else [n_params(K, d, c) for K, d in [(3, 2), (5, 10)] for c in ['spherical', 'diag', 'tied', 'full']]

_ref = []
for _K, _d in [(3, 2), (5, 10)]:
    _X = rng.normal(size=(200, _d))
    for _c in ['spherical', 'diag', 'tied', 'full']:
        _ref.append(GaussianMixture(_K, covariance_type=_c, random_state=0).fit(_X)._n_parameters())
check('parameter counts', ours, _ref)

<details>
<summary><b>💡 Hint</b></summary>

All types share $K-1$ weights and $Kd$ means. Covariances: spherical $K$, diag $Kd$, tied $d(d+1)/2$, full $Kd(d+1)/2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

For $K=3,d=2$: 11, 14, 11, 17. For $K=5,d=10$: 59, 104, 109, 329.
With $d=100$, full needs $K\cdot5050$ covariance parameters, impossible to estimate from 500 points (and singular). Use `diag` or `spherical`, reduce dimension first with [[PCA]], or use `tied`.

```python
def n_params(K, d, cov_type):
    cov = {'spherical': K, 'diag': K * d, 'tied': d * (d + 1) // 2, 'full': K * d * (d + 1) // 2}[cov_type]
    return (K - 1) + K * d + cov

ours = [n_params(K, d, c) for K, d in [(3, 2), (5, 10)] for c in ['spherical', 'diag', 'tied', 'full']]
print(ours)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Gaussian Mixture Models and EM](Gaussian%20Mixture%20Models%20and%20EM.md).*